# Optional ONNX export — 5 extra points

Run the main notebook first. Preserve these cells and insert your implementation
after the export prompt. Do not retrain the model in this notebook.


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import onnx
from sklearn.datasets import load_iris

ACTIVITY_DIR = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
MODELS_DIR = ACTIVITY_DIR / "models"
ONNX_PATH = MODELS_DIR / "iris.onnx"
bundle = joblib.load(MODELS_DIR / "iris.joblib")
model = bundle["model"]
class_names = bundle["class_names"]


## Export the trained model

Use skl2onnx to convert `model` and write `ONNX_PATH`. Use a float32 input named
`X` with shape `(None, 4)`, a numeric label output named `label`, and class IDs
0, 1, 2. Add metadata `class_names` containing `json.dumps(class_names)` so another
runtime can map labels to species. Disable ZipMap to simplify output handling.
Check converter support for your chosen estimator. Do not substitute another model.
Implement the different-runtime application in `extra_src/`; see its README.


In [ ]:
assert ONNX_PATH.is_file()
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
metadata = {item.key: item.value for item in onnx_model.metadata_props}
assert json.loads(metadata["class_names"]) == class_names
input_info = onnx_model.graph.input[0]
assert input_info.name == "X"
assert input_info.type.tensor_type.elem_type == onnx.TensorProto.FLOAT
assert input_info.type.tensor_type.shape.dim[1].dim_value == 4
# The reference evaluator verifies inference without requiring Python ONNX Runtime.
from onnx.reference import ReferenceEvaluator
X = load_iris().data.astype(np.float32)
labels = ReferenceEvaluator(onnx_model).run(["label"], {"X": X})[0]
np.testing.assert_array_equal(np.asarray(labels).reshape(-1), model.predict(X))
